In [ ]:
import time
start_time = time.time()

import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix


In [ ]:
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
print(f"Selected device: {device}")

In [ ]:
model_name = "textattack/distilbert-base-uncased-MRPC"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.to(device)
model.eval()
id2label = model.config.id2label if hasattr(model.config, "id2label") else {0: "not_equivalent", 1: "equivalent"}
label2id = {v: k for k, v in id2label.items()}
positive_label_id = 1 if 1 in id2label else max(id2label.keys())
negative_label_id = 0 if 0 in id2label else min(id2label.keys())
print(f"Loaded model: {model_name}")
print(f"Label mapping: {id2label}")
print(f"Positive label id: {positive_label_id}, Negative label id: {negative_label_id}")

In [ ]:
dataset = load_dataset("glue", "mrpc", split="validation")
print(f"Validation examples: {len(dataset)}")

preview_df = dataset.select(range(min(5, len(dataset)))).to_pandas()
print(preview_df[["sentence1", "sentence2", "label"]].to_string(index=False))

In [ ]:
batch_size = 32
predictions = []
prediction_confidence = []
positive_class_probability = []
negative_class_probability = []
true_labels = []

for start_idx in range(0, len(dataset), batch_size):
    batch = dataset[start_idx:start_idx + batch_size]
    inputs = tokenizer(
        batch["sentence1"],
        batch["sentence2"],
        truncation=True,
        padding=True,
        return_tensors="pt"
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)
        preds = torch.argmax(probs, dim=-1)
        pred_conf = probs.gather(1, preds.unsqueeze(1)).squeeze(1)

    predictions.extend(preds.cpu().tolist())
    prediction_confidence.extend(pred_conf.cpu().tolist())
    positive_class_probability.extend(probs[:, positive_label_id].detach().cpu().tolist())
    negative_class_probability.extend(probs[:, negative_label_id].detach().cpu().tolist())
    true_labels.extend(batch["label"])

print(f"Completed inference for {len(predictions)} examples.")

In [ ]:
accuracy = accuracy_score(true_labels, predictions)
precision, recall, f1, _ = precision_recall_fscore_support(
    true_labels,
    predictions,
    average="binary",
    zero_division=0
)
cm = confusion_matrix(true_labels, predictions, labels=[negative_label_id, positive_label_id])

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print("Confusion Matrix [rows=true, cols=pred] with label order [0, 1]:")
print(cm)

In [ ]:
results_df = pd.DataFrame([
    {
        "model_name": model_name,
        "dataset": "glue/mrpc",
        "split": "validation",
        "num_examples": len(dataset),
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "device": str(device),
        "batch_size": batch_size
    }
])

print(results_df.to_string(index=False))

In [ ]:
examples_df = dataset.to_pandas()[["sentence1", "sentence2", "label"]].copy()
examples_df = examples_df.rename(columns={"label": "true_label"})
examples_df["predicted_label"] = predictions
examples_df["prediction_confidence"] = prediction_confidence
examples_df["positive_class_probability"] = positive_class_probability
examples_df["negative_class_probability"] = negative_class_probability
examples_df["true_label_name"] = examples_df["true_label"].map(id2label)
examples_df["predicted_label_name"] = examples_df["predicted_label"].map(id2label)
examples_df["correct"] = examples_df["true_label"] == examples_df["predicted_label"]
examples_df["error_bucket"] = "correct"
examples_df.loc[(examples_df["true_label"] == negative_label_id) & (examples_df["predicted_label"] == positive_label_id), "error_bucket"] = "false_positive"
examples_df.loc[(examples_df["true_label"] == positive_label_id) & (examples_df["predicted_label"] == negative_label_id), "error_bucket"] = "false_negative"

print(examples_df.head(10).to_string(index=False))

In [ ]:
confidence_summary_df = pd.DataFrame([
    {
        "group": "all_examples",
        "count": len(examples_df),
        "mean_prediction_confidence": examples_df["prediction_confidence"].mean(),
        "median_prediction_confidence": examples_df["prediction_confidence"].median(),
        "min_prediction_confidence": examples_df["prediction_confidence"].min(),
        "max_prediction_confidence": examples_df["prediction_confidence"].max(),
        "mean_positive_class_probability": examples_df["positive_class_probability"].mean()
    },
    {
        "group": "correct_only",
        "count": int(examples_df["correct"].sum()),
        "mean_prediction_confidence": examples_df.loc[examples_df["correct"], "prediction_confidence"].mean(),
        "median_prediction_confidence": examples_df.loc[examples_df["correct"], "prediction_confidence"].median(),
        "min_prediction_confidence": examples_df.loc[examples_df["correct"], "prediction_confidence"].min(),
        "max_prediction_confidence": examples_df.loc[examples_df["correct"], "prediction_confidence"].max(),
        "mean_positive_class_probability": examples_df.loc[examples_df["correct"], "positive_class_probability"].mean()
    },
    {
        "group": "incorrect_only",
        "count": int((~examples_df["correct"]).sum()),
        "mean_prediction_confidence": examples_df.loc[~examples_df["correct"], "prediction_confidence"].mean(),
        "median_prediction_confidence": examples_df.loc[~examples_df["correct"], "prediction_confidence"].median(),
        "min_prediction_confidence": examples_df.loc[~examples_df["correct"], "prediction_confidence"].min(),
        "max_prediction_confidence": examples_df.loc[~examples_df["correct"], "prediction_confidence"].max(),
        "mean_positive_class_probability": examples_df.loc[~examples_df["correct"], "positive_class_probability"].mean()
    }
])

print(confidence_summary_df.to_string(index=False))

In [ ]:
correct_df = examples_df[examples_df["correct"]].copy()
incorrect_df = examples_df[~examples_df["correct"]].copy()
false_positive_df = examples_df[examples_df["error_bucket"] == "false_positive"].copy()
false_negative_df = examples_df[examples_df["error_bucket"] == "false_negative"].copy()

bucket_summary_df = pd.DataFrame([
    {
        "bucket": "correct",
        "count": len(correct_df),
        "mean_prediction_confidence": correct_df["prediction_confidence"].mean(),
        "median_prediction_confidence": correct_df["prediction_confidence"].median(),
        "mean_positive_class_probability": correct_df["positive_class_probability"].mean()
    },
    {
        "bucket": "incorrect",
        "count": len(incorrect_df),
        "mean_prediction_confidence": incorrect_df["prediction_confidence"].mean(),
        "median_prediction_confidence": incorrect_df["prediction_confidence"].median(),
        "mean_positive_class_probability": incorrect_df["positive_class_probability"].mean()
    },
    {
        "bucket": "false_positive",
        "count": len(false_positive_df),
        "mean_prediction_confidence": false_positive_df["prediction_confidence"].mean(),
        "median_prediction_confidence": false_positive_df["prediction_confidence"].median(),
        "mean_positive_class_probability": false_positive_df["positive_class_probability"].mean()
    },
    {
        "bucket": "false_negative",
        "count": len(false_negative_df),
        "mean_prediction_confidence": false_negative_df["prediction_confidence"].mean(),
        "median_prediction_confidence": false_negative_df["prediction_confidence"].median(),
        "mean_positive_class_probability": false_negative_df["positive_class_probability"].mean()
    }
])

print(bucket_summary_df.to_string(index=False))

In [ ]:
confidence_by_outcome_df = (
    examples_df.groupby(["correct", "true_label", "predicted_label"], as_index=False)
    .agg(
        count=("prediction_confidence", "size"),
        mean_prediction_confidence=("prediction_confidence", "mean"),
        median_prediction_confidence=("prediction_confidence", "median"),
        mean_positive_class_probability=("positive_class_probability", "mean")
    )
    .sort_values(["correct", "true_label", "predicted_label"])
)

print(confidence_by_outcome_df.to_string(index=False))

In [ ]:
high_confidence_errors_df = incorrect_df.sort_values("prediction_confidence", ascending=False).copy()
low_confidence_correct_df = correct_df.sort_values("prediction_confidence", ascending=True).copy()

print("Top high-confidence errors:")
if len(high_confidence_errors_df) > 0:
    print(high_confidence_errors_df[["sentence1", "sentence2", "true_label_name", "predicted_label_name", "prediction_confidence", "positive_class_probability", "error_bucket"]].head(10).to_string(index=False))
else:
    print("No errors found.")

print("\nTop low-confidence correct predictions:")
if len(low_confidence_correct_df) > 0:
    print(low_confidence_correct_df[["sentence1", "sentence2", "true_label_name", "predicted_label_name", "prediction_confidence", "positive_class_probability", "error_bucket"]].head(10).to_string(index=False))
else:
    print("No correct predictions found.")

In [ ]:
print("False positives sample:")
if len(false_positive_df) > 0:
    print(false_positive_df[["sentence1", "sentence2", "true_label_name", "predicted_label_name", "prediction_confidence", "positive_class_probability"]].sort_values("prediction_confidence", ascending=False).head(10).to_string(index=False))
else:
    print("No false positives found.")

print("\nFalse negatives sample:")
if len(false_negative_df) > 0:
    print(false_negative_df[["sentence1", "sentence2", "true_label_name", "predicted_label_name", "prediction_confidence", "positive_class_probability"]].sort_values("prediction_confidence", ascending=False).head(10).to_string(index=False))
else:
    print("No false negatives found.")

In [ ]:
error_bucket_counts_df = examples_df["error_bucket"].value_counts().rename_axis("error_bucket").reset_index(name="count")
true_label_counts_df = examples_df["true_label"].value_counts().sort_index().rename_axis("true_label").reset_index(name="count")
pred_label_counts_df = examples_df["predicted_label"].value_counts().sort_index().rename_axis("predicted_label").reset_index(name="count")

print(error_bucket_counts_df.to_string(index=False))
print(true_label_counts_df.to_string(index=False))
print(pred_label_counts_df.to_string(index=False))

In [ ]:
elapsed_seconds = time.time() - start_time
print(f"Total runtime (seconds): {elapsed_seconds:.2f}")